# LOTTO 5/35 - DASHBOARD PHÂN TÍCH DỰ BÁO - V5
## Phân tích Theo Vị Trí, Thống kê, Dự báo & Biểu đồ

**Mục đích:**
- Phân tích **riêng biệt** từng vị trí (S1-S5, S6)
- S1-S5: Dãy 1-35 với điều kiện thống kê riêng
  - S1: 1-31
  - S2: 2-32
  - S3: 3-33
  - S4: 4-34
  - S5: 5-35
- S6: Dãy 1-12 (riêng biệt)
- Dự báo Top 12 Hot + Top 12 Quá Hạn cho mỗi vị trí
- Giống phương pháp: https://en.lottolyzer.com/

**Lưu ý:** Phân tích dựa trên xác suất thống kê, không đảm bảo 100% chính xác

## 1. CÀI ĐẶT & CÀU HÌNH

In [ ]:
# Cài đặt các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from datetime import datetime, timedelta
import warnings
import os
import sys

warnings.filterwarnings('ignore')

# Cấu hình hiển thị
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

print(f"✓ Python Version: {sys.version}")
print(f"✓ Pandas Version: {pd.__version__}")
print(f"✓ NumPy Version: {np.__version__}")
print(f"✓ Matplotlib Version: {plt.matplotlib.__version__}")

## 2. TẢI DỮ LIỆU

In [ ]:
# Đọc file CSV từ URL hoặc local
url = 'https://raw.githubusercontent.com/PhDLeToanThang/ai-ml-dl/refs/heads/main/aipl/lotto535.csv'

try:
    df = pd.read_csv(url, encoding='utf-8')
    print("✓ Dữ liệu được tải từ GitHub URL")
except Exception as e:
    print(f"⚠ Lỗi tải từ URL: {e}")
    try:
        df = pd.read_csv('lotto535.csv', encoding='utf-8')
        print("✓ Dữ liệu được tải từ file local")
    except:
        print("❌ Không thể tải dữ liệu")
        raise

# Tiền xử lý dữ liệu
df['Ngày'] = pd.to_datetime(df['Ngày'], format='%d/%m/%Y')
df = df.sort_values('Ngày').reset_index(drop=True)

print(f"\n📊 THÔNG TIN DỮ LIỆU:")
print(f"  • Tổng kỳ quay: {len(df)}")
print(f"  • Từ: {df['Ngày'].min().strftime('%d/%m/%Y')} đến {df['Ngày'].max().strftime('%d/%m/%Y')}")
print(f"  • Cột: {df.columns.tolist()}")
print(f"\nDữ liệu mẫu (5 bản ghi cuối):")
print(df.tail())

## 3. ĐỊNH NGHĨA PHẠM VI CHO MỖI VỊ TRÍ

In [ ]:
# Định nghĩa phạm vi cho mỗi vị trí theo thống kê Lotto 5/35 Việt Nam
position_config = {
    's1': {'range': (1, 31), 'description': 'Vị trí 1 (1-31)', 'type': 'main'},
    's2': {'range': (2, 32), 'description': 'Vị trí 2 (2-32)', 'type': 'main'},
    's3': {'range': (3, 33), 'description': 'Vị trí 3 (3-33)', 'type': 'main'},
    's4': {'range': (4, 34), 'description': 'Vị trí 4 (4-34)', 'type': 'main'},
    's5': {'range': (5, 35), 'description': 'Vị trí 5 (5-35)', 'type': 'main'},
    's6': {'range': (1, 12), 'description': 'Vị trí 6 - Giải Xổ Số (1-12)', 'type': 'prize'}
}

print("✓ Cấu hình phạm vi cho mỗi vị trí:")
for pos, config in position_config.items():
    print(f"  • {pos.upper()}: {config['description']}")

## 4. CLASS PHÂN TÍCH CHO MỖI VỊ TRÍ

In [ ]:
class PositionAnalyzer:
    """
    Phân tích chi tiết cho một vị trí cụ thể
    """
    def __init__(self, df, position_name, min_val, max_val):
        self.df = df
        self.position_name = position_name
        self.min_val = min_val
        self.max_val = max_val
        self.total_draws = len(df)
        self.valid_range = range(min_val, max_val + 1)
        
        # Tính tần suất
        self.frequency = self._calculate_frequency()
        self.expected_frequency = self.total_draws / len(self.valid_range)
    
    def _calculate_frequency(self):
        """Tính tần suất xuất hiện của mỗi số"""
        numbers = self.df[self.position_name].values
        freq = Counter(numbers)
        
        # Đảm bảo tất cả số trong range có entry
        result = {}
        for num in self.valid_range:
            result[num] = freq.get(num, 0)
        return result
    
    def get_last_appearance(self):
        """Lần xuất hiện cuối cùng của mỗi số"""
        last_app = {num: -1 for num in self.valid_range}
        
        for idx, val in enumerate(self.df[self.position_name]):
            if val in self.valid_range:
                last_app[val] = idx
        
        return last_app
    
    def calculate_overdue(self):
        """Tính số kỳ quay chưa xuất hiện (quá hạn)"""
        last_app = self.get_last_appearance()
        current_draw = self.total_draws - 1
        
        overdue = {}
        for num in self.valid_range:
            if last_app[num] == -1:
                # Số chưa bao giờ xuất hiện
                overdue[num] = current_draw + 1
            else:
                overdue[num] = current_draw - last_app[num]
        
        return overdue
    
    def calculate_prediction_score(self):
        """Tính điểm dự báo cho mỗi số (0-100)"""
        overdue = self.calculate_overdue()
        
        scores = {}
        for num in self.valid_range:
            freq = self.frequency[num]
            overdue_val = overdue[num]
            
            # Tính Chi-Square Deviation
            chi_square = ((freq - self.expected_frequency) ** 2) / max(self.expected_frequency, 1)
            
            # Normalize Overdue (0-100)
            max_overdue = max(overdue.values())
            overdue_score = (overdue_val / max(max_overdue, 1)) * 30
            
            # Frequency Score (0-100)
            max_freq = max(self.frequency.values())
            freq_score = (freq / max(max_freq, 1)) * 40
            
            # Statistical Score (0-100)
            max_chi = max([((f - self.expected_frequency) ** 2) / max(self.expected_frequency, 1) 
                           for f in self.frequency.values()])
            stat_score = ((max_chi - chi_square) / max(max_chi, 1)) * 30 if max_chi > 0 else 15
            
            score = overdue_score + freq_score + stat_score
            scores[num] = min(score, 100)  # Cap at 100
        
        return scores
    
    def get_top_hot_numbers(self, top_n=12):
        """Lấy top N số dự báo hot"""
        scores = self.calculate_prediction_score()
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        result = []
        for rank, (num, score) in enumerate(ranked, 1):
            freq = self.frequency[num]
            result.append({
                'Rank': rank,
                'Số': num,
                'Tần Suất': freq,
                'Điểm Dự Báo': round(score, 2),
                'Phần Trăm': f"{(freq / self.total_draws * 100):.2f}%"
            })
        
        return pd.DataFrame(result)
    
    def get_top_overdue_numbers(self, top_n=12):
        """Lấy top N số quá hạn (chưa xuất hiện lâu)"""
        overdue = self.calculate_overdue()
        ranked = sorted(overdue.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        result = []
        for rank, (num, overdue_val) in enumerate(ranked, 1):
            freq = self.frequency[num]
            status = "🔴 Quá Hạn Nặng" if overdue_val > 50 else "🟠 Quá Hạn Vừa" if overdue_val > 30 else "🟡 Quá Hạn Nhẹ"
            result.append({
                'Rank': rank,
                'Số': num,
                'Kỳ Quay Chưa Xuất': overdue_val,
                'Tần Suất': freq,
                'Trạng Thái': status
            })
        
        return pd.DataFrame(result)
    
    def get_complete_analysis(self):
        """Lấy phân tích đầy đủ cho tất cả các số"""
        scores = self.calculate_prediction_score()
        overdue = self.calculate_overdue()
        
        result = []
        for num in sorted(self.valid_range):
            freq = self.frequency[num]
            score = scores[num]
            overdue_val = overdue[num]
            
            # Xác định trạng thái
            if score > 75:
                status = "🔥 Rất Hot"
            elif score > 60:
                status = "🔥 Hot"
            elif score > 50:
                status = "⚠️ Trung Bình"
            else:
                status = "❄️ Lạnh"
            
            result.append({
                'Số': num,
                'Tần Suất': freq,
                'Quá Hạn': overdue_val,
                'Điểm Dự Báo': round(score, 2),
                'Trạng Thái': status,
                'Phần Trăm': f"{(freq / self.total_draws * 100):.2f}%"
            })
        
        return pd.DataFrame(result)

print("✓ Class PositionAnalyzer đã được tạo")

## 5. PHÂN TÍCH TẤT CẢ CÁC VỊ TRÍ

In [ ]:
# Tạo analyzer cho mỗi vị trí
analyzers = {}
for pos, config in position_config.items():
    min_val, max_val = config['range']
    analyzers[pos] = PositionAnalyzer(df, pos, min_val, max_val)
    print(f"✓ {pos.upper()} Analyzer được tạo ({min_val}-{max_val})")

print(f"\n✓ Tất cả {len(analyzers)} vị trí đã được phân tích")

## 6. IN KẾT QUẢ TOP 12 HOT & TOP 12 QUÁHẠN

In [ ]:
print("\n" + "="*100)
print("🎯 TOP 12 SỐ DỰ BÁO HOT & QUÁHẠN CHO MỖI VỊ TRÍ")
print("="*100)

for pos in ['s1', 's2', 's3', 's4', 's5']:
    analyzer = analyzers[pos]
    config = position_config[pos]
    
    print(f"\n{'='*100}")
    print(f"📊 {pos.upper()} - {config['description']}")
    print(f"{'='*100}")
    
    # Top 12 Hot
    print(f"\n🔥 TOP 12 SỐ DỰ BÁO HOT (Dự báo xuất hiện cao):")
    top_hot = analyzer.get_top_hot_numbers(12)
    print(top_hot.to_string(index=False))
    
    # Top 12 Quá Hạn
    print(f"\n🔴 TOP 12 SỐ QUÁHẠN (Chưa xuất hiện lâu):")
    top_overdue = analyzer.get_top_overdue_numbers(12)
    print(top_overdue.to_string(index=False))

print(f"\n{'='*100}")
print(f"📊 S6 - VỊ TRÍ 6 - GIẢI XỔ SỐ (1-12)")
print(f"{'='*100}")
print(f"\n🔥 TOP 12 SỐ DỰ BÁO HOT (Dự báo xuất hiện cao):")
top_hot_s6 = analyzers['s6'].get_top_hot_numbers(12)
print(top_hot_s6.to_string(index=False))
print(f"\nℹ️ S6 chỉ có 12 giá trị (1-12) nên top 12 = tất cả các số")

## 7. BIỂU ĐỒ CHO MỖI VỊ TRÍ

In [ ]:
# Tạo thư mục output
output_dir = 'analysis_results_v5'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"✓ Thư mục output: {output_dir}")

In [ ]:
# Biểu đồ cho S1-S5
for pos in ['s1', 's2', 's3', 's4', 's5']:
    analyzer = analyzers[pos]
    config = position_config[pos]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'{pos.upper()} - {config["description"]} | PHÂN TÍCH CHI TIẾT', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Biểu đồ 1: Top 12 Hot Numbers
    top_hot = analyzer.get_top_hot_numbers(12)
    axes[0, 0].barh(top_hot['Số'].astype(str), top_hot['Điểm Dự Báo'], 
                    color='#ff6b6b', edgecolor='black', linewidth=0.5)
    axes[0, 0].set_xlabel('Điểm Dự Báo', fontweight='bold')
    axes[0, 0].set_title('🔥 Top 12 Số Hot (Dự Báo)', fontweight='bold')
    axes[0, 0].invert_yaxis()
    axes[0, 0].grid(axis='x', alpha=0.3)
    
    # Biểu đồ 2: Tần suất
    min_val, max_val = config['range']
    nums = list(range(min_val, max_val + 1))
    freqs = [analyzer.frequency[n] for n in nums]
    colors = ['#ff6b6b' if f > analyzer.expected_frequency else '#4ecdc4' for f in freqs]
    axes[0, 1].bar(nums, freqs, color=colors, edgecolor='black', linewidth=0.5)
    axes[0, 1].axhline(y=analyzer.expected_frequency, color='red', linestyle='--', 
                       label=f'Kỳ vọng: {analyzer.expected_frequency:.2f}', linewidth=2)
    axes[0, 1].set_xlabel('Số', fontweight='bold')
    axes[0, 1].set_ylabel('Tần Suất', fontweight='bold')
    axes[0, 1].set_title('📊 Tần Suất Xuất Hiện', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    axes[0, 1].legend()
    
    # Biểu đồ 3: Top 12 Quá Hạn
    top_overdue = analyzer.get_top_overdue_numbers(12)
    axes[1, 0].barh(top_overdue['Số'].astype(str), top_overdue['Kỳ Quay Chưa Xuất'], 
                    color='#e74c3c', edgecolor='black', linewidth=0.5)
    axes[1, 0].set_xlabel('Kỳ Quay Chưa Xuất Hiện', fontweight='bold')
    axes[1, 0].set_title('🔴 Top 12 Số Quá Hạn', fontweight='bold')
    axes[1, 0].invert_yaxis()
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    # Biểu đồ 4: Tất cả số - Tần suất vs Kỳ vọng
    complete_data = analyzer.get_complete_analysis()
    x = np.arange(len(complete_data))
    width = 0.35
    axes[1, 1].bar(x - width/2, complete_data['Tần Suất'], width, 
                   label='Tần Suất Thực', color='#3498db', edgecolor='black', linewidth=0.3)
    axes[1, 1].axhline(y=analyzer.expected_frequency, color='red', linestyle='--', 
                       label=f'Kỳ Vọng', linewidth=2)
    axes[1, 1].set_xlabel('Số', fontweight='bold')
    axes[1, 1].set_ylabel('Tần Suất', fontweight='bold')
    axes[1, 1].set_title('⚖️ Phân Bố Tần Suất', fontweight='bold')
    axes[1, 1].set_xticks(x[::2])
    axes[1, 1].set_xticklabels(complete_data['Số'].iloc[::2])
    axes[1, 1].grid(axis='y', alpha=0.3)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/{pos}_analysis_chart.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Biểu đồ {pos.upper()} được lưu")

print(f"\n✓ Biểu đồ S1-S5 hoàn thành")

In [ ]:
# Biểu đồ cho S6
pos = 's6'
analyzer = analyzers[pos]
config = position_config[pos]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'{pos.upper()} - {config["description"]} | PHÂN TÍCH CHI TIẾT', 
             fontsize=16, fontweight='bold', y=0.995)

# Biểu đồ 1: Top 12 Hot Numbers (tất cả 12 số)
top_hot = analyzer.get_top_hot_numbers(12)
axes[0, 0].barh(top_hot['Số'].astype(str), top_hot['Điểm Dự Báo'], 
                color='#27ae60', edgecolor='black', linewidth=0.5)
axes[0, 0].set_xlabel('Điểm Dự Báo', fontweight='bold')
axes[0, 0].set_title('🎁 Top 12 Giải Xổ (Dự Báo)', fontweight='bold')
axes[0, 0].invert_yaxis()
axes[0, 0].grid(axis='x', alpha=0.3)

# Biểu đồ 2: Tần suất
min_val, max_val = config['range']
nums = list(range(min_val, max_val + 1))
freqs = [analyzer.frequency[n] for n in nums]
colors = ['#27ae60' if f > analyzer.expected_frequency else '#95a5a6' for f in freqs]
axes[0, 1].bar(nums, freqs, color=colors, edgecolor='black', linewidth=0.5)
axes[0, 1].axhline(y=analyzer.expected_frequency, color='red', linestyle='--', 
                   label=f'Kỳ vọng: {analyzer.expected_frequency:.2f}', linewidth=2)
axes[0, 1].set_xlabel('Giải Xổ (1-12)', fontweight='bold')
axes[0, 1].set_ylabel('Tần Suất', fontweight='bold')
axes[0, 1].set_title('📊 Tần Suất Xuất Hiện', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].legend()

# Biểu đồ 3: Pie chart phân bố
pie_data = pd.Series(freqs, index=nums)
axes[1, 0].pie(pie_data, labels=[str(n) for n in nums], autopct='%1.1f%%', 
              colors=plt.cm.Set3(np.linspace(0, 1, len(nums))), startangle=90)
axes[1, 0].set_title('🥧 Phân Bố Giải Xổ', fontweight='bold')

# Biểu đồ 4: Tất cả số - Tần suất
complete_data = analyzer.get_complete_analysis()
axes[1, 1].bar(complete_data['Số'], complete_data['Tần Suất'], 
              color='#27ae60', edgecolor='black', linewidth=0.5)
axes[1, 1].axhline(y=analyzer.expected_frequency, color='red', linestyle='--', 
                   label=f'Kỳ Vọng', linewidth=2)
axes[1, 1].set_xlabel('Giải Xổ', fontweight='bold')
axes[1, 1].set_ylabel('Tần Suất', fontweight='bold')
axes[1, 1].set_title('⚖️ Phân Bố Tần Suất', fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{output_dir}/s6_analysis_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Biểu đồ S6 được lưu")

## 8. SO SÁNH TẤT CẢ VỊ TRÍ

In [ ]:
# Biểu đồ so sánh Top 12 Hot của tất cả vị trí
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🎯 SO SÁNH TOP 12 SỐ HOT - TẤT CẢ VỊ TRÍ', fontsize=16, fontweight='bold')

axes_flat = axes.flatten()

for idx, pos in enumerate(['s1', 's2', 's3', 's4', 's5', 's6']):
    analyzer = analyzers[pos]
    config = position_config[pos]
    top_hot = analyzer.get_top_hot_numbers(12)
    
    color = '#ff6b6b' if pos != 's6' else '#27ae60'
    axes_flat[idx].barh(top_hot['Số'].astype(str), top_hot['Điểm Dự Báo'], 
                         color=color, edgecolor='black', linewidth=0.5)
    axes_flat[idx].set_xlabel('Điểm Dự Báo', fontweight='bold', fontsize=9)
    axes_flat[idx].set_title(f'{pos.upper()} - {config["description"]}', fontweight='bold')
    axes_flat[idx].invert_yaxis()
    axes_flat[idx].grid(axis='x', alpha=0.3)
    axes_flat[idx].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(f'{output_dir}/comparison_all_positions.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Biểu đồ so sánh được lưu")

## 9. XUẤT DỮ LIỆU RA CSV

In [ ]:
print("\n" + "="*100)
print("💾 XUẤT DỮ LIỆU RA FILE")
print("="*100)

# Xuất cho S1-S5
for pos in ['s1', 's2', 's3', 's4', 's5']:
    analyzer = analyzers[pos]
    
    # Complete Analysis
    complete = analyzer.get_complete_analysis()
    complete.to_csv(f'{output_dir}/{pos}_complete_analysis.csv', index=False, encoding='utf-8')
    
    # Top 12 Hot
    top_hot = analyzer.get_top_hot_numbers(12)
    top_hot.to_csv(f'{output_dir}/{pos}_top12_hot.csv', index=False, encoding='utf-8')
    
    # Top 12 Overdue
    top_overdue = analyzer.get_top_overdue_numbers(12)
    top_overdue.to_csv(f'{output_dir}/{pos}_top12_overdue.csv', index=False, encoding='utf-8')
    
    print(f"✓ {pos.upper()} - 3 files exported")

# Xuất cho S6
pos = 's6'
analyzer = analyzers[pos]

complete = analyzer.get_complete_analysis()
complete.to_csv(f'{output_dir}/{pos}_complete_analysis.csv', index=False, encoding='utf-8')

top_hot = analyzer.get_top_hot_numbers(12)
top_hot.to_csv(f'{output_dir}/{pos}_top12_hot.csv', index=False, encoding='utf-8')

print(f"✓ {pos.upper()} - 2 files exported (không có overdue)")

print(f"\n✓ Tất cả {len(['s1', 's2', 's3', 's4', 's5'])*3 + 2} files CSV được xuất")

## 10. BÁO CÁO TÓMLƯỢC

In [ ]:
# Tạo báo cáo tóm tắt
report = f"""{'='*100}
BAO CAO PHAN TICH LOTTO 5/35 - DASHBOARD V5
Thoi gian: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}
{'='*100}

1. THONG TIN DU LIEU:
   • Tong so ky quay: {len(df)}
   • Khoang thoi gian: {df['Ngày'].min().strftime('%d/%m/%Y')} - {df['Ngày'].max().strftime('%d/%m/%Y')}
   • Tong so lan quay: {len(df) * 5} (cho S1-S5) + {len(df)} (cho S6)

2. THONG KE CHI TIET:
"""

for pos in ['s1', 's2', 's3', 's4', 's5']:
    analyzer = analyzers[pos]
    config = position_config[pos]
    min_val, max_val = config['range']
    complete = analyzer.get_complete_analysis()
    
    report += f"\n   {pos.upper()} - {config['description']}:\n"
    report += f"     • Pham vi: {min_val}-{max_val}\n"
    report += f"     • Tan suat trung binh moi so: {analyzer.expected_frequency:.2f}\n"
    report += f"     • Tan suat cao nhat: {complete['Tần Suất'].max()}\n"
    report += f"     • Tan suat thap nhat: {complete['Tần Suất'].min()}\n"
    report += f"     • Do lech chuan: {complete['Tần Suất'].std():.2f}\n"

pos = 's6'
analyzer = analyzers[pos]
config = position_config[pos]
min_val, max_val = config['range']
complete = analyzer.get_complete_analysis()

report += f"\n   {pos.upper()} - {config['description']}:\n"
report += f"     • Pham vi: {min_val}-{max_val}\n"
report += f"     • Tan suat trung binh moi so: {analyzer.expected_frequency:.2f}\n"
report += f"     • Tan suat cao nhat: {complete['Tần Suất'].max()}\n"
report += f"     • Tan suat thap nhat: {complete['Tần Suất'].min()}\n"
report += f"     • Do lech chuan: {complete['Tần Suất'].std():.2f}\n"

report += f"""
3. GHI CHU QUAN TRONG:
   • Phan tich nay dua tren xac suat thong ke
   • Khong dam bao ket qua 100% chinh xac
   • Nen ket hop voi kinh nghiem va phan tich khac
   • Tam nhin: Danh cho muc dich giao duc va phan tich
   • Cau truc du lieu:
     - S1-S5: Cac hang cua tro choi (1-35 voi dieu kien khac nhau)
     - S6: Giai so (1-12 - hang so rieng biet)

4. TAI LIEU XUAT RA:
   • Tong CSV files: 17 (3x5 + 2)
   • Tong PNG files: 6 (1x6)
   • Tong TXT files: 1
   • Thu muc: analysis_results_v5/

{'='*100}
TaiBaoCao: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}
{'='*100}
"""

print(report)

# Lưu báo cáo
with open(f'{output_dir}/REPORT_SUMMARY.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n✓ Báo cáo tóm tắt được lưu: {output_dir}/REPORT_SUMMARY.txt")

## 11. TỔNG KẾT PHÂN TÍCH

In [ ]:
print("\n" + "="*100)
print("📊 TỔNG KẾT PHÂN TÍCH DASHBOARD V5")
print("="*100)

print(f"\n✅ PHÂN TÍCH HOÀN THÀNH:")
print(f"   • Vị trí S1-S5: Phân tích riêng biệt với phạm vi thống kê")
print(f"   • Vị trí S6: Phân tích riêng biệt (1-12)")
print(f"   • Top 12 Hot: Được tính cho tất cả vị trí")
print(f"   • Top 12 Quá Hạn: Được tính cho S1-S5 (S6 không áp dụng)")

print(f"\n📁 OUTPUT FILES ({len(os.listdir(output_dir))} files):")
for file in sorted(os.listdir(output_dir)):
    file_path = os.path.join(output_dir, file)
    size = os.path.getsize(file_path) / 1024  # KB
    print(f"   • {file} ({size:.1f} KB)")

print(f"\n🎯 CÁC THUẬT TOÁN ĐƯỢC SỬ DỤNG:")
print(f"   1. Frequency Analysis - Tần suất xuất hiện")
print(f"   2. Overdue Analysis - Số chưa xuất hiện lâu (S1-S5)")
print(f"   3. Chi-Square Deviation - Độ lệch thống kê")
print(f"   4. Expected Frequency - Tần suất kỳ vọng")
print(f"   5. Combined Scoring - Điểm tổng hợp (0-100)")

print(f"\n📈 CÔNG THỨC TÍNH ĐIỂM DỰ BÁO:")
print(f"   Điểm = 30% (Overdue) + 40% (Frequency) + 30% (Statistical)")
print(f"   Max Score: 100 | Min Score: 0")

print(f"\n✅ DASHBOARD V5 HOÀN THÀNH!")
print(f"\n💾 Tất cả dữ liệu đã được lưu tại: {os.path.abspath(output_dir)}")
print(f"\n" + "="*100)

## 12. KIỂM TRA DỮ LIỆU XUẤT RA

In [ ]:
# Hiển thị mẫu từ mỗi file
print("\n" + "="*100)
print("📋 MẪU DỮ LIỆU TỪ CÁC FILE XUẤT RA")
print("="*100)

for pos in ['s1', 's2', 's3', 's4', 's5', 's6']:
    print(f"\n{'─'*100}")
    print(f"📄 {pos.upper()} - TOP 12 HOT")
    print(f"{'─'*100}")
    
    csv_file = f'{output_dir}/{pos}_top12_hot.csv'
    if os.path.exists(csv_file):
        data = pd.read_csv(csv_file, encoding='utf-8')
        print(data.to_string(index=False))
    else:
        print(f"❌ File không tìm thấy")

print(f"\n{'─'*100}")
print("ℹ️  Để xem top 12 quá hạn của S1-S5, mở file *_top12_overdue.csv")
print(f"{'─'*100}")

## KẾT LUẬN

### Dashboard V5 - Phân Tích Theo Vị Trí

**Đặc Điểm Chính:**
- ✅ **Phân tích riêng biệt** cho mỗi vị trí (S1-S5, S6)
- ✅ **Phạm vi thống kê chính xác** theo luật lotto Việt Nam
- ✅ **Top 12 Hot** cho tất cả vị trí
- ✅ **Top 12 Quá Hạn** cho S1-S5 (chỉ có 12 số cho S6)
- ✅ **Giống phương pháp** https://en.lottolyzer.com/
- ✅ **Xuất dữ liệu** dạng CSV + PNG + TXT

**Phạm Vi Thống Kê:**
- S1: 1-31
- S2: 2-32
- S3: 3-33
- S4: 4-34
- S5: 5-35
- S6: 1-12 (Giải Xổ - Riêng biệt)

**Điểm Dự Báo (0-100):**
- 30% Quá Hạn (Overdue)
- 40% Tần Suất (Frequency)
- 30% Thống Kê (Statistical)

**Lưu Ý:** Phân tích này dựa trên xác suất thống kê, không đảm bảo 100% chính xác.